# Deep Learning Models for Intrusion Detection
## CICIDS2017 Dataset

This notebook trains 9 Deep Learning models:
- 6 Supervised models (SIDS)
- 3 Anomaly Detection models (AIDS)

## Step 1: Import Libraries

In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models

# TabNet
!pip install pytorch-tabnet
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

print("All libraries imported successfully!")

All libraries imported successfully!


## Step 2: Load Dataset

In [3]:
# Load the CICIDS2017 dataset
df = pd.read_csv("../cicids2017_full.csv")
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
df.head()

Dataset shape: (2827876, 81)

First few rows:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,source_file,binary_label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,Monday-WorkingHours.pcap_ISCX.csv,0
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,Monday-WorkingHours.pcap_ISCX.csv,0
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,Monday-WorkingHours.pcap_ISCX.csv,0
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,Monday-WorkingHours.pcap_ISCX.csv,0
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,0.0,0,0,0.0,0.0,0,0,BENIGN,Monday-WorkingHours.pcap_ISCX.csv,0


## Step 3: Prepare Data

In [4]:
# Shuffle the data
df_sample = df.sample(frac=1, random_state=42)

# Separate features and labels
universal_cols = [
    'Flow Duration',
    'Destination Port',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Flow Packets/s',
    'Flow Bytes/s',
    'Fwd Packet Length Mean',
    'Fwd Packet Length Max',
    'Bwd Packet Length Mean',
    'Packet Length Mean',
    'Packet Length Std',
    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow IAT Max',
    'SYN Flag Count',
    'ACK Flag Count',
    'RST Flag Count',
    'Average Packet Size'
]

X = df[universal_cols]
y = df["binary_label"]

print("Features shape:", X.shape)
print("Labels shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts())

Features shape: (2827876, 18)
Labels shape: (2827876,)

Class distribution:
binary_label
0    2271320
1     556556
Name: count, dtype: int64


## Step 4: Split Data

In [5]:
# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

Training set size: 2262300
Test set size: 565576


## Step 5: Normalize Features

In [6]:
# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data normalized successfully!")

Data normalized successfully!


---
# PART 1: Supervised Deep Learning Models (SIDS)
---

## Model 1: MLP (Multi-Layer Perceptron)

In [7]:
print("Building MLP Model...")

mlp = models.Sequential([
    layers.Dense(128, activation="relu", input_shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

mlp.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining MLP...")
mlp.fit(
    X_train_scaled, y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

y_pred_mlp = (mlp.predict(X_test_scaled) > 0.5).astype(int)

print("MLP (SIDS)")
print(classification_report(y_test, y_pred_mlp))

Building MLP Model...


c:\Users\dhruv\.venvs\ai-ml\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training MLP...
Epoch 1/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.9536 - loss: 0.1121 - val_accuracy: 0.9589 - val_loss: 0.0960
Epoch 2/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 36s 3ms/step - accuracy: 0.9605 - loss: 0.0892 - val_accuracy: 0.9628 - val_loss: 0.0793
Epoch 3/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - accuracy: 0.9660 - loss: 0.0765 - val_accuracy: 0.9682 - val_loss: 0.0660
Epoch 4/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9740 - loss: 0.0626 - val_accuracy: 0.9717 - val_loss: 0.0586
Epoch 5/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 20s 2ms/step - accuracy: 0.9818 - loss: 0.0493 - val_accuracy: 0.9849 - val_loss: 0.0420
Epoch 6/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9855 - loss: 0.0404 - val_accuracy: 0.9883 - val_loss: 0.0335
Epoch 7/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9875 - loss: 0.0351 - val_accuracy: 0.9837 - val_loss: 0.0386
Epoch 8/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 17s 2ms/step - accuracy:

## Model 2: TabNet

In [8]:
print("Building TabNet Model...")

tabnet = TabNetClassifier(
    n_d=16,
    n_a=16,
    n_steps=5,
    gamma=1.5,
    device_name="cuda" if torch.cuda.is_available() else "cpu"
)

print("\nTraining TabNet...")
tabnet.fit(
    X_train_scaled, y_train.values,
    eval_set=[(X_test_scaled, y_test.values)],
    max_epochs=30,
    patience=5,
    batch_size=1024
)

y_pred_tabnet = tabnet.predict(X_test_scaled)

print("TabNet (SIDS)")
print(classification_report(y_test, y_pred_tabnet))

Building TabNet Model...

Training TabNet...


c:\Users\dhruv\.venvs\ai-ml\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.11402 | val_0_auc: 0.83625 |  0:05:31s
epoch 1  | loss: 0.08362 | val_0_auc: 0.82191 |  0:11:04s
epoch 2  | loss: 0.06371 | val_0_auc: 0.85833 |  0:15:35s
epoch 3  | loss: 0.06394 | val_0_auc: 0.90172 |  0:19:59s
epoch 4  | loss: 0.0835  | val_0_auc: 0.95964 |  0:24:36s
epoch 5  | loss: 0.06186 | val_0_auc: 0.95783 |  0:28:55s
epoch 6  | loss: 0.04791 | val_0_auc: 0.90205 |  0:33:14s
epoch 7  | loss: 0.05056 | val_0_auc: 0.96145 |  0:37:45s
epoch 8  | loss: 0.04526 | val_0_auc: 0.93159 |  0:42:28s
epoch 9  | loss: 0.05268 | val_0_auc: 0.79116 |  0:46:47s
epoch 10 | loss: 0.0496  | val_0_auc: 0.88629 |  0:51:05s
epoch 11 | loss: 0.04671 | val_0_auc: 0.9078  |  0:55:48s
epoch 12 | loss: 0.03962 | val_0_auc: 0.89241 |  1:00:12s

Early stopping occurred at epoch 12 with best_epoch = 7 and best_val_0_auc = 0.96145


c:\Users\dhruv\.venvs\ai-ml\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


TabNet (SIDS)
              precision    recall  f1-score   support

           0       0.92      0.96      0.94    454265
           1       0.80      0.66      0.72    111311

    accuracy                           0.90    565576
   macro avg       0.86      0.81      0.83    565576
weighted avg       0.90      0.90      0.90    565576



## Model 3: Transformer (Multi-Head Attention)

In [9]:
print("Building Transformer Model...")

inputs = layers.Input(shape=(X_train_scaled.shape[1],))
x = layers.Dense(128, activation="relu")(inputs)
x = layers.Reshape((1, 128))(x)

attn = layers.MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
x = layers.Flatten()(attn)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

transformer = models.Model(inputs, outputs)

transformer.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining Transformer...")
transformer.fit(
    X_train_scaled, y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

y_pred_trans = (transformer.predict(X_test_scaled) > 0.5).astype(int)

print("Transformer (SIDS)")
print(classification_report(y_test, y_pred_trans))

Building Transformer Model...

Training Transformer...
Epoch 1/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 141s 20ms/step - accuracy: 0.9548 - loss: 0.1083 - val_accuracy: 0.9622 - val_loss: 0.0901
Epoch 2/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 143s 20ms/step - accuracy: 0.9627 - loss: 0.0840 - val_accuracy: 0.9583 - val_loss: 0.0857
Epoch 3/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 143s 20ms/step - accuracy: 0.9769 - loss: 0.0571 - val_accuracy: 0.9779 - val_loss: 0.0629
Epoch 4/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 142s 20ms/step - accuracy: 0.9831 - loss: 0.0445 - val_accuracy: 0.9850 - val_loss: 0.0343
Epoch 5/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 152s 21ms/step - accuracy: 0.9843 - loss: 0.0412 - val_accuracy: 0.9851 - val_loss: 0.0368
Epoch 6/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 149s 21ms/step - accuracy: 0.9854 - loss: 0.0378 - val_accuracy: 0.9864 - val_loss: 0.0331
Epoch 7/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 142s 20ms/step - accuracy: 0.9861 - loss: 0.0358 - val_accuracy: 0.9869 - val_loss: 0.0378
Epoch 8/10
707

## Model 4: Vanilla LSTM

In [10]:
print("Building Vanilla LSTM Model...")

# Reshape for LSTM (samples, timesteps, features)
X_train_seq = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_seq = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

lstm_model = models.Sequential([
    layers.LSTM(64, activation="relu", input_shape=(1, X_train_scaled.shape[1])),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining Vanilla LSTM...")
lstm_model.fit(
    X_train_seq, y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

y_pred_lstm = (lstm_model.predict(X_test_seq) > 0.5).astype(int)

print("Vanilla LSTM (SIDS)")
print(classification_report(y_test, y_pred_lstm))

Building Vanilla LSTM Model...

Training Vanilla LSTM...
Epoch 1/10


c:\Users\dhruv\.venvs\ai-ml\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


7070/7070 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - accuracy: 0.9526 - loss: 0.1159 - val_accuracy: 0.9593 - val_loss: 0.0917
Epoch 2/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - accuracy: 0.9611 - loss: 0.0867 - val_accuracy: 0.9651 - val_loss: 0.0817
Epoch 3/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - accuracy: 0.9652 - loss: 0.0766 - val_accuracy: 0.9686 - val_loss: 0.0707
Epoch 4/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 29s 4ms/step - accuracy: 0.9696 - loss: 0.0685 - val_accuracy: 0.9768 - val_loss: 0.0617
Epoch 5/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - accuracy: 0.9758 - loss: 0.0589 - val_accuracy: 0.9809 - val_loss: 0.0537
Epoch 6/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 24s 3ms/step - accuracy: 0.9818 - loss: 0.0495 - val_accuracy: 0.9831 - val_loss: 0.0535
Epoch 7/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - accuracy: 0.9848 - loss: 0.0426 - val_accuracy: 0.9863 - val_loss: 0.0371
Epoch 8/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - accuracy: 0.9865 - loss: 0.0381 - val

## Model 5: Stacked LSTM

In [11]:
print("Building Stacked LSTM Model...")

stacked_lstm = models.Sequential([
    layers.LSTM(64, activation="relu", return_sequences=True, input_shape=(1, X_train_scaled.shape[1])),
    layers.LSTM(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

stacked_lstm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining Stacked LSTM...")
stacked_lstm.fit(
    X_train_seq, y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

y_pred_stacked = (stacked_lstm.predict(X_test_seq) > 0.5).astype(int)

print("Stacked LSTM (SIDS)")
print(classification_report(y_test, y_pred_stacked))

Building Stacked LSTM Model...

Training Stacked LSTM...
Epoch 1/10


c:\Users\dhruv\.venvs\ai-ml\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


7070/7070 ━━━━━━━━━━━━━━━━━━━━ 31s 4ms/step - accuracy: 0.9516 - loss: 0.1171 - val_accuracy: 0.9598 - val_loss: 0.0970
Epoch 2/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - accuracy: 0.9614 - loss: 0.0880 - val_accuracy: 0.9641 - val_loss: 0.0821
Epoch 3/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - accuracy: 0.9664 - loss: 0.0760 - val_accuracy: 0.9694 - val_loss: 0.0637
Epoch 4/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.9753 - loss: 0.0612 - val_accuracy: 0.9814 - val_loss: 0.0507
Epoch 5/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - accuracy: 0.9820 - loss: 0.0499 - val_accuracy: 0.9858 - val_loss: 0.0416
Epoch 6/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.9852 - loss: 0.0409 - val_accuracy: 0.9846 - val_loss: 0.0362
Epoch 7/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 31s 4ms/step - accuracy: 0.9868 - loss: 0.0362 - val_accuracy: 0.9892 - val_loss: 0.0342
Epoch 8/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - accuracy: 0.9878 - loss: 0.0366 - val

## Model 6: Bidirectional LSTM

In [12]:
print("Building Bidirectional LSTM Model...")

bilstm = models.Sequential([
    layers.Bidirectional(layers.LSTM(64, activation="relu"), input_shape=(1, X_train_scaled.shape[1])),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

bilstm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining Bidirectional LSTM...")
bilstm.fit(
    X_train_seq, y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

y_pred_bilstm = (bilstm.predict(X_test_seq) > 0.5).astype(int)

print("Bidirectional LSTM (SIDS)")
print(classification_report(y_test, y_pred_bilstm))

Building Bidirectional LSTM Model...

Training Bidirectional LSTM...
Epoch 1/10


c:\Users\dhruv\.venvs\ai-ml\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


7070/7070 ━━━━━━━━━━━━━━━━━━━━ 31s 4ms/step - accuracy: 0.9534 - loss: 0.1125 - val_accuracy: 0.9598 - val_loss: 0.0920
Epoch 2/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.9617 - loss: 0.0865 - val_accuracy: 0.9646 - val_loss: 0.0771
Epoch 3/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 29s 4ms/step - accuracy: 0.9685 - loss: 0.0722 - val_accuracy: 0.9772 - val_loss: 0.0669
Epoch 4/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 29s 4ms/step - accuracy: 0.9768 - loss: 0.0587 - val_accuracy: 0.9833 - val_loss: 0.0464
Epoch 5/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 29s 4ms/step - accuracy: 0.9833 - loss: 0.0463 - val_accuracy: 0.9847 - val_loss: 0.0381
Epoch 6/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 27s 4ms/step - accuracy: 0.9861 - loss: 0.0391 - val_accuracy: 0.9877 - val_loss: 0.0333
Epoch 7/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.9872 - loss: 0.0351 - val_accuracy: 0.9885 - val_loss: 0.0300
Epoch 8/10
7070/7070 ━━━━━━━━━━━━━━━━━━━━ 28s 4ms/step - accuracy: 0.9881 - loss: 0.0321 - val

---
# PART 2: Anomaly Detection Models (AIDS)
---

## Model 7: AutoEncoder

In [13]:
print("Building AutoEncoder Model...")

# Train only on benign data
X_train_benign = X_train_scaled[y_train == 0]

ae = models.Sequential([
    layers.Dense(64, activation="relu", input_shape=(X_train_benign.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(X_train_benign.shape[1])
])

ae.compile(optimizer="adam", loss="mse")

print("\nTraining AutoEncoder...")
ae.fit(
    X_train_benign, X_train_benign,
    epochs=20,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

recon = ae.predict(X_test_scaled)
recon_error = np.mean(np.square(X_test_scaled - recon), axis=1)

threshold = np.percentile(recon_error[y_test == 0], 95)
y_pred_ae = (recon_error > threshold).astype(int)

print("AutoEncoder (AIDS)")
print(classification_report(y_test, y_pred_ae))

Building AutoEncoder Model...

Training AutoEncoder...


c:\Users\dhruv\.venvs\ai-ml\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0359 - val_loss: 0.0095
Epoch 2/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0108 - val_loss: 0.0128
Epoch 3/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0066 - val_loss: 0.0072
Epoch 4/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0056 - val_loss: 5.4713e-04
Epoch 5/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.0053 - val_loss: 9.4952e-04
Epoch 6/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.0041 - val_loss: 0.0036
Epoch 7/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.0040 - val_loss: 0.0296
Epoch 8/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.0050 - val_loss: 2.4187e-04
Epoch 9/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 0.0044 - val_loss: 0.0023
Epoch 10/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - loss: 0.0037 - val_loss: 0.0010
Epoch 11/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 15s 2ms/step - loss: 0.0030 - val_loss: 0.0031
Epoch 1

## Model 8: LSTM AutoEncoder

In [14]:
print("Building LSTM AutoEncoder Model...")

# Reshape for LSTM
X_train_lstm = X_train_benign.reshape((X_train_benign.shape[0], 1, X_train_benign.shape[1]))
X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

lstm_ae = models.Sequential([
    layers.LSTM(64, activation="relu", input_shape=(1, X_train_benign.shape[1]), return_sequences=True),
    layers.LSTM(32, activation="relu", return_sequences=False),
    layers.RepeatVector(1),
    layers.LSTM(32, activation="relu", return_sequences=True),
    layers.LSTM(64, activation="relu", return_sequences=True),
    layers.TimeDistributed(layers.Dense(X_train_benign.shape[1]))
])

lstm_ae.compile(optimizer="adam", loss="mse")

print("\nTraining LSTM AutoEncoder...")
lstm_ae.fit(
    X_train_lstm, X_train_lstm,
    epochs=20,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

recon_lstm = lstm_ae.predict(X_test_lstm)
lstm_error = np.mean(np.square(X_test_lstm - recon_lstm), axis=(1, 2))

threshold = np.percentile(lstm_error[y_test == 0], 95)
y_pred_lstm = (lstm_error > threshold).astype(int)

print("LSTM AutoEncoder (AIDS)")
print(classification_report(y_test, y_pred_lstm))

Building LSTM AutoEncoder Model...

Training LSTM AutoEncoder...


c:\Users\dhruv\.venvs\ai-ml\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
12777/12777 ━━━━━━━━━━━━━━━━━━━━ 59s 4ms/step - loss: 0.0611 - val_loss: 0.0117
Epoch 2/20
12777/12777 ━━━━━━━━━━━━━━━━━━━━ 53s 4ms/step - loss: 0.0233 - val_loss: 0.0191
Epoch 3/20
12777/12777 ━━━━━━━━━━━━━━━━━━━━ 51s 4ms/step - loss: 0.0186 - val_loss: 0.0016
Epoch 4/20
12777/12777 ━━━━━━━━━━━━━━━━━━━━ 52s 4ms/step - loss: 0.0136 - val_loss: 0.0311
Epoch 5/20
12777/12777 ━━━━━━━━━━━━━━━━━━━━ 52s 4ms/step - loss: 0.0146 - val_loss: 0.0099
Epoch 6/20
12777/12777 ━━━━━━━━━━━━━━━━━━━━ 53s 4ms/step - loss: 0.0143 - val_loss: 0.0729
Epoch 7/20
12777/12777 ━━━━━━━━━━━━━━━━━━━━ 52s 4ms/step - loss: 0.0172 - val_loss: 0.0637
Epoch 8/20
12777/12777 ━━━━━━━━━━━━━━━━━━━━ 53s 4ms/step - loss: 0.0168 - val_loss: 0.0012
Epoch 9/20
12777/12777 ━━━━━━━━━━━━━━━━━━━━ 54s 4ms/step - loss: 0.0181 - val_loss: 0.0210
Epoch 10/20
12777/12777 ━━━━━━━━━━━━━━━━━━━━ 54s 4ms/step - loss: 0.0140 - val_loss: 0.0360
Epoch 11/20
12777/12777 ━━━━━━━━━━━━━━━━━━━━ 53s 4ms/step - loss: 0.0167 - val_loss: 0.00

## Model 9: Denoising AutoEncoder

In [15]:
print("Building Denoising AutoEncoder Model...")

# Add noise to training data
noise_factor = 0.1
X_train_noisy = X_train_benign + noise_factor * np.random.normal(size=X_train_benign.shape)

dae = models.Sequential([
    layers.Dense(64, activation="relu", input_shape=(X_train_benign.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(X_train_benign.shape[1])
])

dae.compile(optimizer="adam", loss="mse")

print("\nTraining Denoising AutoEncoder...")
dae.fit(
    X_train_noisy, X_train_benign,  # Train to denoise
    epochs=20,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

recon = dae.predict(X_test_scaled)
recon_error = np.mean(np.square(X_test_scaled - recon), axis=1)

threshold = np.percentile(recon_error[y_test == 0], 95)
y_pred_dae = (recon_error > threshold).astype(int)

print("Denoising AutoEncoder (AIDS)")
print(classification_report(y_test, y_pred_dae))

Building Denoising AutoEncoder Model...


c:\Users\dhruv\.venvs\ai-ml\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training Denoising AutoEncoder...
Epoch 1/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - loss: 0.0360 - val_loss: 0.0090
Epoch 2/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - loss: 0.0140 - val_loss: 0.0043
Epoch 3/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0112 - val_loss: 0.0028
Epoch 4/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0090 - val_loss: 0.0026
Epoch 5/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - loss: 0.0065 - val_loss: 0.0074
Epoch 6/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0067 - val_loss: 0.0078
Epoch 7/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0057 - val_loss: 0.0026
Epoch 8/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0052 - val_loss: 0.0123
Epoch 9/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0071 - val_loss: 0.0030
Epoch 10/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0041 - val_loss: 0.0029
Epoch 11/20
6389/6389 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - loss: 0.0053 - v

---
# Save Models
---

In [16]:
# Save Keras models
mlp.save('../models/mlp_model.h5')
transformer.save('../models/transformer_model.h5')
lstm_model.save('../models/vanilla_lstm_model.h5')
stacked_lstm.save('../models/stacked_lstm_model.h5')
bilstm.save('../models/bidirectional_lstm_model.h5')
ae.save('../models/autoencoder_model.h5')
lstm_ae.save('../models/lstm_autoencoder_model.h5')
dae.save('../models/denoising_autoencoder_model.h5')

# Save TabNet
tabnet.save_model('../models/tabnet_model')

print("All 9 DL models saved successfully!")

Successfully saved model at ../models/tabnet_model.zip
All 9 DL models saved successfully!


---
# Summary
---

We trained 9 Deep Learning models:

**Supervised Learning (6 models):**
1. MLP (Multi-Layer Perceptron)
2. TabNet (Attention-based)
3. Transformer (Multi-Head Attention)
4. Vanilla LSTM
5. Stacked LSTM
6. Bidirectional LSTM

**Anomaly Detection (3 models):**
7. AutoEncoder
8. LSTM AutoEncoder
9. Denoising AutoEncoder